# Hw2.3 - Аналіз книг Amazon Top-50

In [1]:
import pandas as pdimport numpy as npimport matplotlib.pyplot as pltimport seaborn as sns%matplotlib inlinesns.set_style('whitegrid')plt.rcParams['figure.figsize'] = (12, 6)plt.rcParams['font.size'] = 10

In [2]:
df = pd.read_csv('bestsellers with categories.csv')df.columns = ['name', 'author', 'user_rating', 'reviews', 'price', 'year', 'genre']print(f'Dataset size: {len(df)} books')

**Відповідь: Про скільки книг зберігає дані датасет? - 550**

In [3]:
print(f'Missing values: {df.isnull().sum().sum()}')

**Відповідь: Чи є в якихосх змінних пропуски? - Ні**

In [4]:
print(f'Unique genres: {df["genre"].unique().tolist()}')

**Відповідь: Які є унікальні жанри? - Fiction, Non Fiction**

In [5]:
print(f'Price - Max: {df["price"].max()}, Min: {df["price"].min()}, Mean: {df["price"].mean():.2f}, Median: {df["price"].median():.2f}')

**Відповідь: Максимальна ціна? - 105****Відповідь: Мінімальна ціна? - 0****Відповідь: Середня ціна? - 13.10****Відповідь: Медіанна ціна? - 11.00**

In [6]:
highest = df['user_rating'].max()count = len(df[df['user_rating'] == highest])most_reviewed = df.loc[df['reviews'].idxmax()]print(f'Highest rating: {highest}, Count: {count}')print(f'Most reviewed: {most_reviewed["name"]} ({most_reviewed["reviews"]} reviews)')

**Відповідь: Який рейтинг у датасеті найвищий? - 4.9****Відповідь: Скільки книг мають такий рейтинг? - 52****Відповідь: Яка книга має найбільше відгуків? - Where the Crawdads Sing**

In [7]:
books_2015 = df[df['year'] == 2015]most_expensive = books_2015.loc[books_2015['price'].idxmax()]fiction_2010 = len(df[(df['genre'] == 'Fiction') & (df['year'] == 2010)])rating_49 = len(df[(df['user_rating'] == 4.9) & (df['year'].isin([2010, 2011]))])print(f'Most expensive in 2015: {most_expensive["name"]} - ${most_expensive["price"]}')print(f'Fiction books in 2010: {fiction_2010}')print(f'Books with 4.9 rating in 2010-2011: {rating_49}')

**Відповідь: З тих книг, що потрапили до Топ-50 у 2015 році, яка книга найдорожча? - Publication Manual of the American Psychological Association, 6th Edition****Відповідь: Скільки книг жанру Fiction потрапили до Топ-50 у 2010 році? - 20****Відповідь: Скільки книг з рейтингом 4.9 потрапило до рейтингу у 2010 та 2011 роках? - 1**

In [8]:
books_2015_cheap = df[(df['year'] == 2015) & (df['price'] < 8)].sort_values('price')if len(books_2015_cheap) > 0:    last_book = books_2015_cheap.iloc[-1]['name']    print(f'Last book in sorted list: {last_book}')

**Відповідь: Яка книга остання у відсортованому списку? - Old School (Diary of a Wimpy Kid #10)**

In [9]:
genre_prices = df.groupby('genre')['price'].agg(['min', 'max'])print(f'Fiction - Max: {genre_prices.loc["Fiction", "max"]}, Min: {genre_prices.loc["Fiction", "min"]}')print(f'Non Fiction - Max: {genre_prices.loc["Non Fiction", "max"]}, Min: {genre_prices.loc["Non Fiction", "min"]}')

**Відповідь: Максимальна ціна для жанру Fiction: 82****Відповідь: Мінімальна ціна для жанру Fiction: 0****Відповідь: Максимальна ціна для жанру Non Fiction: 105****Відповідь: Мінімальна ціна для жанру Non Fiction: 0**

In [10]:
author_counts = df.groupby('author').size().reset_index(name='count').sort_values('count', ascending=False)print(f'Author table shape: {author_counts.shape}')top_author = author_counts.iloc[0]print(f'Top author: {top_author["author"]} - {top_author["count"]} books')

**Відповідь: Якого розмірності вийшла таблиця? - (248, 1)****Відповідь: Який автор має найбільше книг? - Jeff Kinney****Відповідь: Скільки книг цього автора? - 12**

In [11]:
author_ratings = df.groupby('author')['user_rating'].mean().reset_index(name='avg_rating').sort_values('avg_rating')lowest = author_ratings.iloc[0]print(f'Lowest avg rating: {lowest["author"]} - {lowest["avg_rating"]:.2f}')

**Відповідь: У якого автора середній рейтинг мінімальний? - Donna Tartt****Відповідь: Який у цього автора середній рейтинг? - 3.90**

In [12]:
merged = pd.concat([author_counts.set_index('author'), author_ratings.set_index('author')], axis=1)merged = merged.sort_values(['count', 'avg_rating'])print(f'First author in sorted list: {merged.index[0]}')

**Відповідь: Який автор перший у списку? - Muriel Barbery**

## Графік 1: Розподіл цін книг

In [13]:
fig, ax = plt.subplots(figsize=(12, 6))ax.hist(df['price'], bins=25, color='#3498db', edgecolor='black', alpha=0.7)ax.axvline(df['price'].mean(), color='red', linestyle='--', linewidth=2, label=f'Середнє: ${df["price"].mean():.2f}')ax.axvline(df['price'].median(), color='green', linestyle=':', linewidth=2, label=f'Медіана: ${df["price"].median():.2f}')ax.set_xlabel('Ціна ($)', fontsize=12)ax.set_ylabel('Кількість книг', fontsize=12)ax.set_title('Розподіл цін книг Amazon Top-50', fontsize=14, fontweight='bold')ax.legend(fontsize=10)ax.grid(axis='y', alpha=0.3)plt.tight_layout()plt.show()

## Графік 2: Книги за жанрами

In [14]:
fig, ax = plt.subplots(figsize=(8, 8))genre_counts = df['genre'].value_counts()colors = ['#e74c3c', '#3498db']explode = (0.05, 0)wedges, texts, autotexts = ax.pie(genre_counts.values, labels=genre_counts.index, autopct='%1.1f%%',                                       colors=colors, explode=explode, startangle=90)ax.set_title('Розподіл книг за жанрами', fontsize=14, fontweight='bold')for autotext in autotexts:    autotext.set_fontsize(12)    autotext.set_fontweight('bold')plt.tight_layout()plt.show()

## Графік 3: Рейтинг книг

In [15]:
fig, ax = plt.subplots(figsize=(10, 6))rating_counts = df['user_rating'].value_counts().sort_index()colors = plt.cm.RdYlGn(np.linspace(0.2, 0.9, len(rating_counts)))bars = ax.bar(rating_counts.index.astype(str), rating_counts.values, color=colors, edgecolor='black')ax.set_xlabel('Рейтинг', fontsize=12)ax.set_ylabel('Кількість книг', fontsize=12)ax.set_title('Розподіл рейтингів книг', fontsize=14, fontweight='bold')ax.grid(axis='y', alpha=0.3)for bar, val in zip(bars, rating_counts.values):    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, str(val),            ha='center', va='bottom', fontsize=9)plt.tight_layout()plt.show()

## Графік 4: Середня ціна по роках

In [16]:
fig, ax = plt.subplots(figsize=(12, 6))yearly_prices = df.groupby('year')['price'].agg(['mean', 'std'])ax.errorbar(yearly_prices.index, yearly_prices['mean'], yerr=yearly_prices['std'],           marker='o', markersize=8, capsize=5, capthick=2, linewidth=2, color='#9b59b6')ax.fill_between(yearly_prices.index, yearly_prices['mean'] - yearly_prices['std'],                yearly_prices['mean'] + yearly_prices['std'], alpha=0.2, color='#9b59b6')ax.set_xlabel('Рік', fontsize=12)ax.set_ylabel('Середня ціна ($)', fontsize=12)ax.set_title('Середня ціна книг по роках (з стандартним відхиленням)', fontsize=14, fontweight='bold')ax.grid(True, alpha=0.3)plt.tight_layout()plt.show()

## Графік 5: Топ-10 авторів за кількістю книг

In [17]:
top_10_authors = author_counts.head(10)fig, ax = plt.subplots(figsize=(12, 6))colors = plt.cm.viridis(np.linspace(0.2, 0.8, len(top_10_authors)))bars = ax.barh(range(len(top_10_authors)), top_10_authors['count'].values, color=colors, edgecolor='black')ax.set_yticks(range(len(top_10_authors)))ax.set_yticklabels(top_10_authors['author'].values)ax.set_xlabel('Кількість книг', fontsize=12)ax.set_ylabel('Автор', fontsize=12)ax.set_title('Топ-10 авторів за кількістю книг у Top-50', fontsize=14, fontweight='bold')ax.grid(axis='x', alpha=0.3)ax.invert_yaxis()for bar, val in zip(bars, top_10_authors['count'].values):    ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2, str(val),            va='center', fontsize=10)plt.tight_layout()plt.show()